=================================================
### Práctica 4b — Detección y Seguimiento con YOLOv8
=================================================

##### Este cuaderno realiza:
1. Detección y seguimiento de personas y vehículos con **YOLOv8**.
2. Detección de matrículas con un modelo propio entrenado.
3. Lectura de matrículas
4. Generación de un **CSV con detecciones y tracking**.


### Carga de modelos y configuración inicial

In [44]:
from ultralytics import YOLO
import cv2, csv
from collections import defaultdict, deque
import pandas as pd

# Archivos de entrada y salida
video_input = "C0142.mp4"
video_output = "p4_output.mp4"
csv_output   = "p4_results.csv"

# Modelos YOLO
general_model = YOLO('yolo11n.pt').to('cuda')  # Detección general (GPU)
plate_model   = YOLO('yolo_runs/plates_detection/weights/best.pt').to('cuda')  # Detección de matrículas (GPU)

# Clases relevantes
classNames = ["person", "bicycle", "car", "motorbike", "bus", "truck"]
general_classes = [0,1,2,3,4,5]

# Diccionario de colores (BGR)
class_colors = {
    "person": (255, 255, 0),  # cian
    "bicycle": (0, 255, 0),
    "car": (0, 0, 255),
    "motorbike": (255, 0, 0),
    "bus": (0, 165, 255),
    "truck": (128, 0, 128)
}
plate_color = (0, 255, 255)  # amarillo

# Para que cada objeto se cuente UNA vez:
counted_ids = {k: set() for k in ["person","bicycle","car","motorbike","bus","truck","plate"]}

# Para mantener la clase "canónica" del track_id (la primera clase que tuvo)
track_classes = {}   # track_id -> clase_asignada

# Contadores en tiempo real (únicos)
total_by_classes = defaultdict(int)  # por clase (person, car, ...)

### Preparación del vídeo

In [45]:
# Carga del vídeo
vid = cv2.VideoCapture(video_input)
width  = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = vid.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid = cv2.VideoWriter(video_output, fourcc, fps, (width, height))

# Historial de tracking
track_history = defaultdict(lambda: deque(maxlen=5))

### Funciones auxiliares para el CSV

In [46]:
def write_csv_header(path):
    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["frame","tipo_objeto","confianza","track_id","x1","y1","x2","y2",
                         "plate","plate_conf","mx1","my1","mx2","my2"])

def append_detection(path, frame, obj_class, conf, track_id,
                     x1, y1, x2, y2, plate, plate_conf, mx1, my1, mx2, my2):
    with open(path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([frame,obj_class,conf,track_id,x1,y1,x2,y2,
                         plate,plate_conf,mx1,my1,mx2,my2])

write_csv_header(csv_output)

In [ ]:
import pytesseract

def readPlate (img):
    #cv2.imwrite("testimg.jpg",img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    #cv2.imwrite("testtresh.jpg",thresh)

    items = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = items[0] if len(items) == 2 else items[1]

    img_contour = img.copy()
    
    for i in range(len(contours)):
        area = cv2.contourArea(contours[i])
        if 2 < area < 10000:
            cv2.drawContours(img_contour, contours, i, (0, 0, 255), 1)
    
    detected = ""
    for c in reversed(contours):
        x, y, w, h = cv2.boundingRect(c)
        ratio = h/w
        area = cv2.contourArea(c)
        base = np.ones(thresh.shape, dtype=np.uint8)
        if ratio > 0.9 and 2 < area < 10000:
            base[y:y+h, x:x+w] = thresh[y:y+h, x:x+w]
            segment = cv2.bitwise_not(base)
            custom_config = r'-l spa --oem 3 --psm 10 '
            c = pytesseract.image_to_string(segment, config=custom_config)
            detected = detected + c

    if detected == "":
        return "None"
    else:
        return detected.replace("\n","")


### Procesamiento frame a frame con detección, seguimiento y anonimización

In [ ]:
frame_n = 0

while True:
    ret, frame = vid.read()
    if not ret:
        break
    frame_n += 1

    results = general_model.track(frame, persist=True, classes=general_classes)
    current_ids = set()

    # Iterar detecciones del frame
    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            track_id = int(box.id[0]) if box.id is not None else -1
            if track_id == -1 or cls >= len(classNames):
                continue

            detected_class = classNames[cls]
            current_ids.add(track_id)

            # Mantener clase consistente por track_id
            if track_id not in track_classes:
                track_classes[track_id] = detected_class
                # Contar solo una vez por track_id
                if track_id not in counted_ids.get(detected_class, set()):
                    counted_ids.setdefault(detected_class, set()).add(track_id)
                    total_by_classes[detected_class] += 1

            clase = track_classes.get(track_id, detected_class)
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Dibujar bbox y etiqueta
            color = class_colors.get(clase, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"[{track_id}] {clase} {conf:.2f}", 
                        (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                        0.6, color, 2, cv2.LINE_AA)

            # Registrar detección en CSV
            append_detection(csv_output, frame_n, clase, conf, track_id, x1, y1, x2, y2,
                             "", "", "", "", "", "")

    prev_ids = current_ids.copy()

    # ----------------------------
    # Mostrar conteos en pantalla (sin fondo)
    # ----------------------------
    y0 = 25
    delta = 22
    for i, clsname in enumerate(["person","car","motorbike","bus","bicycle","truck","plate"]):
        val = total_by_classes.get(clsname, 0)
        txt = f"{clsname}: {val}"
        cv2.putText(frame, txt, (10, y0 + i*delta),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)

    out_vid.write(frame)

# Fin loop
vid.release()
out_vid.release()
print("CSV de detecciones generado:", csv_output)
print("Video de detecciones generado:", video_output)


0: 384x640 4 cars, 1 bus, 55.6ms
Speed: 57.1ms preprocess, 55.6ms inference, 9.7ms postprocess per image at shape (1, 3, 384, 640)

0: 352x416 (no detections), 18.5ms
Speed: 1.3ms preprocess, 18.5ms inference, 1.0ms postprocess per image at shape (1, 3, 352, 416)

0: 320x416 (no detections), 41.7ms
Speed: 3.1ms preprocess, 41.7ms inference, 3.9ms postprocess per image at shape (1, 3, 320, 416)

0: 288x416 (no detections), 32.0ms
Speed: 2.0ms preprocess, 32.0ms inference, 4.1ms postprocess per image at shape (1, 3, 288, 416)

0: 288x416 (no detections), 31.9ms
Speed: 3.7ms preprocess, 31.9ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 416)

0: 320x416 (no detections), 32.8ms
Speed: 3.9ms preprocess, 32.8ms inference, 5.7ms postprocess per image at shape (1, 3, 320, 416)

0: 384x640 4 cars, 1 bus, 17.8ms
Speed: 3.2ms preprocess, 17.8ms inference, 14.5ms postprocess per image at shape (1, 3, 384, 640)

0: 352x416 1 plate, 63.6ms
Speed: 6.0ms preprocess, 63.6ms inference, 